In [1]:
# ============================================
# Imports
# ============================================

from pathlib import Path
from collections import Counter, defaultdict
import gzip
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# ============================================
# Configuration
# ============================================

# Workflow repository
REPO = Path.home() / "Code" / "processing" / "riboseq"

# Project data directory
PROJECT = Path("/mnt/d/Ibnu/riboseq/AT")

# nf-core output currently used for validation/post-processing
NFCORE_RUN = PROJECT / "nfcore" / "10pct"

# Full biological annotation
GFF3 = REPO / "refs" / "at.gff3.gz"

# Output table directory
TABLES = PROJECT / "TABLES"
TABLES.mkdir(parents=True, exist_ok=True)

# Output file
ANNOTATION_OUT = TABLES / "annotation.10pct.tsv"

In [3]:
# ============================================
# Input check
# ============================================

paths = {
    "REPO": REPO,
    "PROJECT": PROJECT,
    "NFCORE_RUN": NFCORE_RUN,
    "GFF3": GFF3,
    "TABLES": TABLES,
}

for name, path in paths.items():
    print(f"{name:12s}: {path}")
    print(f"{'exists':12s}: {path.exists()}")
    print()

REPO        : /home/ha-ibnu/Code/processing/riboseq
exists      : True

PROJECT     : /mnt/d/Ibnu/riboseq/AT
exists      : True

NFCORE_RUN  : /mnt/d/Ibnu/riboseq/AT/nfcore/10pct
exists      : True

GFF3        : /home/ha-ibnu/Code/processing/riboseq/refs/at.gff3.gz
exists      : True

TABLES      : /mnt/d/Ibnu/riboseq/AT/TABLES
exists      : True



In [4]:
# ============================================
# GFF3 attribute parser
# ============================================

def parse_gff3_attributes(attr: str) -> dict:
    result = {}

    for item in attr.strip().split(";"):
        if not item or "=" not in item:
            continue

        key, value = item.split("=", 1)
        result[key] = value

    return result

In [5]:
# ============================================
# Parse GFF3
# ============================================

records = []

with gzip.open(GFF3, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip().split("\t")

        if len(fields) != 9:
            continue

        chrom, source, feature, start, end, score, strand, phase, attr = fields
        attrs = parse_gff3_attributes(attr)

        records.append({
            "Chr": chrom,
            "Source": source,
            "Feature": feature,
            "Start": int(start),
            "End": int(end),
            "Strand": strand,
            "Phase": phase,
            "ID": attrs.get("ID"),
            "Parent": attrs.get("Parent"),
            "Name": attrs.get("Name"),
        })

gff3 = pd.DataFrame(records)

gff3.head()

,Chr,Source,Feature,Start,End,Strand,Phase,ID,Parent,Name
0,Chr1,TAIR,chromosome,1,30427671,.,.,chromosome:1,None,None
1,Chr2,TAIR,chromosome,1,19698289,.,.,chromosome:2,None,None
2,Chr3,TAIR,chromosome,1,23459830,.,.,chromosome:3,None,None
3,Chr4,TAIR,chromosome,1,18585056,.,.,chromosome:4,None,None
4,Chr5,TAIR,chromosome,1,26975502,.,.,chromosome:5,None,None


In [7]:
gff3.shape

(789897, 10)

In [9]:
# ============================================
# Feature summary
# ============================================

feature_summary = (
    gff3["Feature"]
    .value_counts()
    .rename_axis("Feature")
    .reset_index(name="Count")
)

feature_summary

,Feature,Count
0,CDS,286355
1,exon,200542
2,mRNA,52270
3,protein,48359
4,five_prime_UTR,46895
5,three_prime_UTR,41127
6,transposon_fragment,34856
7,gene,33341
8,transposable_element,31189
9,transposable_element_gene,3901


In [10]:
# ============================================
# Keep transcript model features
# ============================================

KEEP_FEATURES = [
    "mRNA",
    "exon",
    "CDS",
    "five_prime_UTR",
    "three_prime_UTR",
]

model = gff3[gff3["Feature"].isin(KEEP_FEATURES)].copy()

model["Length"] = model["End"] - model["Start"] + 1

model.head()

,Chr,Source,Feature,Start,End,Strand,Phase,ID,Parent,Name,Length
8,Chr1,Araport11,mRNA,3631,5899,+,.,AT1G01010.1,AT1G01010,AT1G01010.1,2269
9,Chr1,Araport11,five_prime_UTR,3631,3759,+,.,AT1G01010:five_prime_UTR:1,AT1G01010.1,NAC001:five_prime_UTR:1,129
10,Chr1,Araport11,exon,3631,3913,+,.,AT1G01010:exon:1,AT1G01010.1,NAC001:exon:1,283
11,Chr1,Araport11,CDS,3760,3913,+,0,AT1G01010:CDS:1,AT1G01010.1,NAC001:CDS:1,154
12,Chr1,Araport11,exon,3996,4276,+,.,AT1G01010:exon:2,AT1G01010.1,NAC001:exon:2,281


In [11]:
model["Feature"].value_counts()

Feature
CDS                286355
exon               200542
mRNA                52270
five_prime_UTR      46895
three_prime_UTR     41127
Name: count, dtype: int64

In [12]:
# ============================================
# Transcript backbone
# ============================================

transcripts = (
    model[model["Feature"] == "mRNA"]
    .copy()
    .rename(columns={
        "ID": "Transcript_ID",
        "Parent": "Gene_ID",
        "Start": "Transcript_start",
        "End": "Transcript_end",
    })
    [[
        "Transcript_ID",
        "Gene_ID",
        "Chr",
        "Strand",
        "Transcript_start",
        "Transcript_end",
    ]]
)

transcripts["Genomic_span_length"] = (
    transcripts["Transcript_end"] - transcripts["Transcript_start"] + 1
)

transcripts.head()

,Transcript_ID,Gene_ID,Chr,Strand,Transcript_start,Transcript_end,Genomic_span_length
8,AT1G01010.1,AT1G01010,Chr1,+,3631,5899,2269
25,AT1G01020.2,AT1G01020,Chr1,-,6788,8737,1950
35,AT1G01020.6,AT1G01020,Chr1,-,6788,8737,1950
45,AT1G01020.1,AT1G01020,Chr1,-,6788,9130,2343
56,AT1G01020.4,AT1G01020,Chr1,-,6788,9130,2343
